# Lab 2c · Uma em mil
**Oficina do Builder · Winovoito** — Bancada 2 · ML aplicado

A **Carteira**, fintech fictícia, tem cerca de uma fraude a cada mil transações. Quando o evento é raro, quase todas as intuições sobre métricas falham.

Neste lab você:
1. vê a armadilha dos 99,9%;
2. usa a curva de precisão e recall;
3. ensina o modelo a dar atenção à classe rara (pesos e subamostragem);
4. escolhe o limiar pela **capacidade** da equipe de revisão;
5. monta o sistema completo: regras + modelo + fila humana.

In [ ]:
# Setup: baixa o material aberto da Oficina (rode uma vez)
REPO_URL = "https://github.com/winovoito/oficina-do-builder-aberto"  # endereço definitivo a confirmar
import os, sys
if not os.path.exists("oficina-do-builder-aberto"):
    !git clone -q {REPO_URL}
sys.path.insert(0, "oficina-do-builder-aberto")
%pip install -q -r oficina-do-builder-aberto/requirements.txt
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 30)

## 1 · Os dados
500 mil transações em 60 dias.

In [ ]:
from oficina.sinteticos import gerar_carteira
from oficina import fraude

df = gerar_carteira(n=500_000, semente=23)
print(df.shape, f"· fraude: {df['fraude'].mean():.3%}")
df.drop(columns="tipo_fraude").head()

## 2 · Dividir no tempo
Treino até o dia 39, teste do dia 40 em diante.

In [ ]:
treino, teste = fraude.dividir(df)
print(f"Treino: {len(treino):,} ({treino['fraude'].sum()} fraudes) · Teste: {len(teste):,} ({teste['fraude'].sum()} fraudes)")

## 3 · A armadilha dos 99,9%

In [ ]:
print(f"\"Nunca é fraude\": acurácia {1 - teste['fraude'].mean():.2%} · recall 0%. Inútil.")

## 4 · Modelo sem ajuste e a curva PR

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay
X = fraude.CARACTERISTICAS
sem = fraude.treinar_sem_ajuste(treino)
print(f"PR-AUC sem ajuste: {fraude.pr_auc(sem, teste):.3f}")
PrecisionRecallDisplay.from_estimator(sem, teste[X], teste["fraude"], name="sem ajuste");

## 5 · Ensinar o modelo a ligar para o raro
Duas técnicas: peso maior para os exemplos de fraude e subamostragem das transações legítimas.

In [ ]:
peso = fraude.treinar_com_peso(treino)
sub = fraude.treinar_subamostrado(treino, proporcao=50)
resultado = {"sem ajuste": fraude.pr_auc(sem, teste), "com peso": fraude.pr_auc(peso, teste),
             "subamostrado": fraude.pr_auc(sub, teste)}
display(pd.Series(resultado, name="PR-AUC").round(3).to_frame())
fig, ax = plt.subplots(figsize=(6, 5))
for nome, m in [("sem ajuste", sem), ("com peso", peso), ("subamostrado", sub)]:
    PrecisionRecallDisplay.from_estimator(m, teste[X], teste["fraude"], ax=ax, name=nome)

## 6 · Limiar pela capacidade
A equipe revisa **200 casos por dia**. Em vez de escolher um número mágico, a fila de cada dia recebe as 200 transações com maior nota.

In [ ]:
modelo = peso if resultado["com peso"] >= resultado["subamostrado"] else sub
so_modelo = fraude.avaliar_fila(modelo, teste)
so_modelo

## 7 · Regras + modelo + fila humana
A partir do dia 45 aparece um **golpe novo**: valor alto em dispositivo novo, em horário comercial. O modelo foi treinado antes dele e quase não o conhece. A equipe de fraude percebeu e criou uma regra.

### Exercício · Escreva a regra
Complete `minha_regra` e compare com a regra de referência (`fraude.regra_golpe_novo`).

In [ ]:
def minha_regra(d):
    # TODO: valor acima de um teto em dispositivo novo vai direto para revisão
    return (d["valor"] > 2_000) & d["dispositivo_novo"]

concorda = (minha_regra(teste) == fraude.regra_golpe_novo(teste)).mean()
print(f"Sua regra concorda com a referência em {concorda:.2%} das transações")
print(f"Casos que a regra manda para a fila por dia: {minha_regra(teste).sum() / teste['dia'].nunique():.1f}")

In [ ]:
sistema = fraude.avaliar_fila(modelo, teste, usar_regras=True)
display(pd.DataFrame({"Só modelo": so_modelo, "Regras + modelo": sistema}).T)
print("Critério do lab: o sistema pega mais fraudes por caso revisado →",
      "✅" if sistema["fraudes_por_caso"] > so_modelo["fraudes_por_caso"] else "❌")

## Reflexão
1. **Falso positivo tem custo:** um cliente honesto com o cartão bloqueado no mercado. Como você mediria esse custo?
2. **Piloto automático humano:** se 95% dos casos da fila são legítimos, o revisor começa a aprovar tudo sem ler. O que você mudaria na fila para evitar isso?
3. O golpe novo só foi pego porque alguém escreveu uma regra. O que precisa acontecer para o **modelo** aprender o golpe? (Pense em retreino e monitoramento, aula 2.6.)